# CS 423 Project: Monocular Obstacle Detection & Distance Estimation (Google Colab Runner)
**Team:** Esma Nehir Vardar (S040236), Ali Raza Mustafa (S026665), Emirhan Tarlabaşı (S033747)

Bu notebook projeyi **Google Colab GPU (T4 / L4 / A100)** üzerinde uçtan uca çalıştırmak için hazırlanmıştır:
1. **Google Drive Bağlantısı:** Eğitilen modeller (`checkpoints/`) ve sonuç tabloları (`results/`) Colab kapansa bile kaybolmaması için Google Drive'a yedeklenir.
2. **Otomatik KITTI İndirme:** KITTI veri seti senin bilgisayarına değil, doğrudan Colab'in yüksek hızlı sunucu diskine (`/content/...`) indirilir ve çıkartılır.
3. **Drive-Sequence Split & LiDAR Projeksiyonu:** Veri sızıntısını (data leakage) önleyen 70/10/20 ayrımı ve 3B LiDAR -> 2B derinlik projeksiyonu yapılır.
4. **3 Pipeline Eğitimi & Ortak Değerlendirme:**
   - **Pipeline 1:** `YOLOv10-Nano` (3 seed) + `Depth Anything V2 (Metric Outdoor)`
   - **Pipeline 2:** `YOLOv8-Nano` (3 seed) + `MiDaS Small (LiDAR Scale-Shift Aligned)`
   - **Pipeline 3:** `HOG + Linear SVM` (Pedestrian) + `Pinhole Camera Model`

## 1. GPU Kontrolü ve Google Drive Bağlantısı

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/CS423_Project_Backup'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f'Google Drive yedekleme klasörü hazır: {DRIVE_BACKUP_DIR}')

## 2. Proje Kodlarını Colab'e Yükleme ve Kütüphaneleri Kurma
* **Seçenek A (GitHub):** Projeyi GitHub'a yüklediyseniz `git clone` ile çekebilirsiniz.
* **Seçenek B (Drive / ZIP):** Proje klasörünü (`src`, `scripts`, `configs`, `requirements.txt`) ZIP yapıp Drive'a veya Colab sol paneline yükleyebilirsiniz.

In [ ]:
# Eğer projeyi GitHub'a koyduysanız aşağıya repo linkinizi yazıp yorumu kaldırın:
# !git clone https://github.com/KULLANICI_ADINIZ/cs423-assistive-navigation.git /content/project
# %cd /content/project

# Veya projeyi Google Drive'ınıza yüklediyseniz / zip olarak attıysanız:
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -qq -y aria2

## 3. Resmi KITTI Veri Setini Doğrudan Colab Diskine İndirme
> **İpucu:**
> - Standart Colab disk alanı ~78-100 GB'tır. `download_kitti.py` scripti her `.zip` dosyasını çıkardıktan sonra `.zip` arşivini otomatik silerek disk alanını korur.
> - Eğer bir oturumda **sadece YOLO ve HOG dedektörlerini eğitmek** istiyorsanız `--skip-velodyne` bayrağını ekleyerek 26 GB'lık LiDAR dosyasını atlayıp 2-3 dakikada indirmeyi tamamlayabilirsiniz.
> - Derinlik modellerini ve final değerlendirmeyi (`04_evaluate_all.py`) çalıştırırken tam komutu (`!python scripts/download_kitti.py`) çalıştırın.

In [ ]:
# Tam KITTI indirmesi (Görüntüler + Etiketler + Kalibrasyon + Devkit + Velodyne LiDAR):
!python scripts/download_kitti.py

# (Alternatif: Sadece dedektör eğitimi için hızlı indirme yapmak isterseniz üsttekini yorum yapıp bunu çalıştırın:)
# !python scripts/download_kitti.py --skip-velodyne

## 4. Veri Hazırlığı: Drive-Sequence Split (70/10/20), LiDAR Projeksiyonu ve YOLO Etiketleri

In [ ]:
!python scripts/01_prepare_data.py

## 5. Pipeline 3 (Klasik Baseline): HOG + Linear SVM Eğitimi ve Drive'a Yedekleme

In [ ]:
!python scripts/02_train_hog.py
!cp -r checkpoints /content/drive/MyDrive/CS423_Project_Backup/

## 6. Pipeline 1 & 2: YOLOv10-Nano ve YOLOv8-Nano Eğitimi (3 Random Seed: 42, 123, 999)
Her eğitim bittiğinde ağırlıklar (`checkpoints/*.pt`) otomatik olarak Google Drive'ınıza kopyalanır.

In [ ]:
# 1. YOLOv10-Nano (3 Seed)
!python scripts/03_train_yolo.py --model yolov10n --seeds 42 123 999 --epochs 50 --batch 16 --device 0
!cp -r checkpoints /content/drive/MyDrive/CS423_Project_Backup/

# 2. YOLOv8-Nano (3 Seed)
!python scripts/03_train_yolo.py --model yolov8n --seeds 42 123 999 --epochs 50 --batch 16 --device 0
!cp -r checkpoints /content/drive/MyDrive/CS423_Project_Backup/

## 7. Ortak Protokol Değerlendirmesi (Test Split), Ablation Study ve Hız (FPS) Benchmark'ı

In [ ]:
# Eğer yeni bir Colab oturumu açtıysanız önce Drive'daki eğitilmiş ağırlıkları geri yükleyin:
!cp -r /content/drive/MyDrive/CS423_Project_Backup/checkpoints/* checkpoints/ 2>/dev/null || true

# 3 Pipeline'ın Test Seti Üzerinde Ortak Değerlendirmesi + Ablation Study:
!python scripts/04_evaluate_all.py --split test --device cuda

# Tek GPU Üzerinde Latency (ms) ve FPS Ölçümü:
!python scripts/05_benchmark_speed.py --warmup 20 --runs 100 --device cuda

# Tüm sonuçları ve grafikleri Google Drive'a yedekle:
!cp -r results /content/drive/MyDrive/CS423_Project_Backup/